# CaloDiffusion Electron Preflight: Reference Counts + Native-Cell Ordering

**Purpose:** check the electron reference HDF5 event counts in the planned energy windows and verify that the 6,480 native cells are mapped consistently by the Dataset 2 XML geometry and the repository's actual `XMLHandler` / `GeomConverter` implementation.

**This is a preflight only.** It does not load the trained checkpoint, run diffusion, generate showers, or alter model weights. A GPU is not required (the geometry converter test uses CPU tensors).

## What this notebook will check
1. Exact repository revision and electron data/config/XML paths.
2. HDF5 structure, total event count, energy units/range, and reference-event counts in three energy windows.
3. Whether each window contains enough distinct events for the proposed calibration/pilot/test split.
4. Dataset 2 geometry: 45 layers × 16 angular bins × 9 radial bins = 6,480 cells.
5. XMLHandler and GeomConverter layer boundaries and bin counts.
6. Artificial-number reshape/unreshape test, and a native-index map CSV.

**Important provenance warning:** the validation notebook's default path is the repository cache `data/calochallenge_ds2.hdf5`. Its name alone does not prove that it is the official held-out `dataset_2_2.hdf5` evaluation file or that its events were unseen during checkpoint training/selection. This notebook reports the exact path and fingerprint but does not infer provenance. Confirm it with your guide before describing the results as an independent benchmark.

The official Dataset 2 description says the challenge provides two files of 100,000 electron showers each, with `dataset_2_1.hdf5` for training and `dataset_2_2.hdf5` for evaluation. The standard HDF5 datasets are `showers` (MeV per voxel) and `incident_energies` (MeV). This may differ from the smaller convenience cache in your repository.

Sources:
- [CaloChallenge Dataset 2 description](https://github.com/CaloChallenge/homepage)
- [Dataset 2 release](https://zenodo.org/records/6366271)
- [Repository validation wrapper](https://github.com/Vishal21501/gen-hep-notebooks/blob/00ebcc3534fc25eee0fe15361483833e0bca34e7/capstone/calodiffusion/calodiff_probe.py)
- [Electron Dataset 2 geometry XML](https://github.com/Vishal21501/gen-hep-notebooks/blob/00ebcc3534fc25eee0fe15361483833e0bca34e7/capstone/calodiffusion/paper_code/CaloChallenge/code/binning_dataset_2.xml)


## Before you run

- Open this notebook in Google Colab.
- Run cells from top to bottom.
- Internet access is needed to obtain the repository/cache if they are not already present.
- The notebook pins the same repository revision used in the photon runs: `00ebcc3534fc25eee0fe15361483833e0bca34e7`.
- By default it inspects the same electron cache path used by your validation notebook. If you have the official `dataset_2_2.hdf5` file or another approved reference file, set `REFERENCE_HDF5_OVERRIDE` in the settings cell to its full path.

The planned initial study split is 300 calibration + 150 pilot + 300 test events **per energy window** (750 distinct events per window). The notebook also reports whether the smaller smoke-light split (80 events) or the validation notebook's larger research split (4,500 events) would fit. It will not silently reduce these numbers.


In [ ]:
# Cell 1 — Settings (edit only if needed)
from pathlib import Path

REPO_URL = "https://github.com/Vishal21501/gen-hep-notebooks.git"
REPO_COMMIT = "00ebcc3534fc25eee0fe15361483833e0bca34e7"
REPO_DIR = Path("/content/electron-preflight-gen-hep-notebooks")
CALO_ROOT = REPO_DIR / "capstone" / "calodiffusion"

# Leave blank to inspect the repository cache used by the current validation wrapper.
# To inspect a separately downloaded official evaluation file, set e.g.
# REFERENCE_HDF5_OVERRIDE = "/content/dataset_2_2.hdf5"
REFERENCE_HDF5_OVERRIDE = ""

# Full-file SHA256 is useful for fixed-reference provenance; may take time on a large HDF5.
COMPUTE_FULL_SHA256 = True

# Standard CaloChallenge HDF5 convention: incident_energies and showers are in MeV.
INPUT_UNIT = "MeV"

# Same electron windows used by the current validation notebook: [low, high) GeV.
ENERGY_WINDOWS_GEV = [(8.0, 12.0), (40.0, 60.0), (80.0, 120.0)]

# Proposed controlled validation sample size per window.
N_CALIBRATION = 300
N_PILOT = 150
N_TEST = 300

# Comparison requirements, also printed for planning.
SMOKE_LIGHT = (32, 16, 32)       # total 80
ORIGINAL_SMOKE = (64, 32, 64)    # total 160
PROPOSED_STUDY = (N_CALIBRATION, N_PILOT, N_TEST)
RESEARCH_PROFILE = (2000, 500, 2000)  # total 4500 in current validation notebook

# Only samples a limited number of shower rows for basic value checks by default.
# Set True to stream through every voxel of every event (more I/O, but thorough).
SCAN_ALL_SHOWER_VALUES = False
SAMPLE_ROWS_TO_CHECK = 256

# Fixed seed used only if the notebook creates a fixed candidate event partition CSV.
PARTITION_SEED = 20261110
MAKE_FIXED_PARTITIONS = True

OUTPUT_DIR = Path("/content/electron_preflight_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Pinned repository revision:", REPO_COMMIT)
print("Energy windows (GeV):", ENERGY_WINDOWS_GEV)
print("Proposed split per window:", PROPOSED_STUDY)
print("Reference override:", REFERENCE_HDF5_OVERRIDE or "repository cache")

Pinned repository revision: 00ebcc3534fc25eee0fe15361483833e0bca34e7
Energy windows (GeV): [(8.0, 12.0), (40.0, 60.0), (80.0, 120.0)]
Proposed split per window: (300, 150, 300)
Reference override: repository cache


In [ ]:
# Cell 2 — Imports and lightweight runtime checks
import os, sys, subprocess, hashlib, json, zipfile, io, math, textwrap
import numpy as np
import pandas as pd
import h5py
import torch
import yaml
import xml.etree.ElementTree as ET
from IPython.display import display

os.environ["CALODIFF_DEVICE"] = "cpu"

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("h5py:", h5py.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available (not required):", torch.cuda.is_available())

Python: 3.13.18
NumPy: 2.4.6
h5py: 3.14.0
PyTorch: 2.14.1+cpu
CUDA available (not required): False


## 1. Obtain the exact source revision

This retrieves the repository code and electron geometry at the same revision used in the previous photon runs. It skips automatic Git LFS downloads, then retrieves **only the electron reference cache** if it is stored as an LFS object. The checkpoint is not downloaded or loaded for this preflight.

In [ ]:
# Cell 3 — Clone/check out the pinned repository revision
clone_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Use a fresh runtime.")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True, env=clone_env
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "sparse-checkout", "set", "capstone/calodiffusion"],
        check=True, env=clone_env
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_COMMIT],
        check=True, env=clone_env
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_COMMIT],
        check=True, env=clone_env
    )

actual_sha = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_sha.lower() != REPO_COMMIT.lower():
    raise RuntimeError(f"Wrong source revision: {actual_sha}; expected {REPO_COMMIT}")

print("PASS: exact source revision", actual_sha)
print("CaloDiffusion source directory:", CALO_ROOT)

PASS: exact source revision 00ebcc3534fc25eee0fe15361483833e0bca34e7
CaloDiffusion source directory: /content/electron-preflight-gen-hep-notebooks/capstone/calodiffusion


In [ ]:
# Cell 4 — Resolve the XML, config, and reference HDF5
XML_PATH = CALO_ROOT / "paper_code/CaloChallenge/code/binning_dataset_2.xml"
CONFIG_PATH = CALO_ROOT / "paper_code/configs/config_dataset2.json"
CACHE_HDF5 = CALO_ROOT / "data/calochallenge_ds2.hdf5"

for required in [XML_PATH, CONFIG_PATH]:
    if not required.is_file():
        raise FileNotFoundError(f"Required source file missing: {required}")

def is_git_lfs_pointer(path):
    path = Path(path)
    if not path.is_file():
        return False
    with path.open("rb") as f:
        return f.read(180).startswith(b"version https://git-lfs.github.com/spec/v1")

def ensure_git_lfs():
    result = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
    if result.returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "lfs", "install", "--local"], check=True)

if REFERENCE_HDF5_OVERRIDE.strip():
    REFERENCE_HDF5 = Path(REFERENCE_HDF5_OVERRIDE).expanduser().resolve()
else:
    REFERENCE_HDF5 = CACHE_HDF5
    if is_git_lfs_pointer(REFERENCE_HDF5):
        ensure_git_lfs()
        lfs_path = "capstone/calodiffusion/data/calochallenge_ds2.hdf5"
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "lfs", "pull", "--include=" + lfs_path, "--exclude="],
            check=True
        )

if not REFERENCE_HDF5.is_file():
    raise FileNotFoundError(
        f"Reference HDF5 not found: {REFERENCE_HDF5}. "
        "Upload/copy the approved HDF5 file and set REFERENCE_HDF5_OVERRIDE."
    )
if is_git_lfs_pointer(REFERENCE_HDF5):
    raise RuntimeError("The selected HDF5 is still a Git LFS pointer, not the actual data file.")
if not h5py.is_hdf5(REFERENCE_HDF5):
    raise ValueError(f"Selected file is not a valid HDF5 file: {REFERENCE_HDF5}")

print("Reference file:", REFERENCE_HDF5)
print("File size MiB:", round(REFERENCE_HDF5.stat().st_size / 2**20, 2))

def sha256_path(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

REFERENCE_SHA256 = sha256_path(REFERENCE_HDF5) if COMPUTE_FULL_SHA256 else None
REFERENCE_FILE_ID = (REFERENCE_SHA256 if REFERENCE_SHA256 else
                     f"{REFERENCE_HDF5.name}:{REFERENCE_HDF5.stat().st_size}")
print("Reference SHA256:", REFERENCE_SHA256 or "not calculated (path + size identifier only)")

Reference file: /content/electron-preflight-gen-hep-notebooks/capstone/calodiffusion/data/calochallenge_ds2.hdf5
File size MiB: 81.14
Reference SHA256: bf73fcfa2269b3e0e0a24d0bccdeb052369fdd5ead2690a598f5e18066c6bd9f


## 2. Inspect reference shower counts

The current validation wrapper defaults to `data/calochallenge_ds2.hdf5`. The HDF5 count reported below is the count in **that exact file** (or the explicit override path). We do not assume it contains 100,000 events merely because the official Dataset 2 release has 100,000 events per file.

The three study windows are `[8,12)`, `[40,60)`, and `[80,120)` GeV. The high boundary is excluded to keep bins non-overlapping. We read the comparatively small incident-energy dataset to count events; large shower arrays are not loaded in full.

In [ ]:
# Cell 5 — HDF5 structure and event-count scan
if INPUT_UNIT != "MeV":
    raise ValueError("This preflight assumes standard CaloChallenge HDF5 energy units of MeV.")

with h5py.File(REFERENCE_HDF5, "r") as h5f:
    keys = list(h5f.keys())
    if not {"showers", "incident_energies"}.issubset(keys):
        raise KeyError(f"Expected HDF5 datasets 'showers' and 'incident_energies'; found {keys}")

    showers_ds = h5f["showers"]
    incident_ds = h5f["incident_energies"]

    shower_shape = tuple(showers_ds.shape)
    energy_shape = tuple(incident_ds.shape)
    shower_dtype = str(showers_ds.dtype)
    energy_dtype = str(incident_ds.dtype)

    if len(shower_shape) != 2:
        raise ValueError(f"Expected flattened showers shape (N,6480); got {shower_shape}")
    if shower_shape[1] != 6480:
        raise ValueError(f"Expected 6480 voxels per electron shower; got {shower_shape[1]}")
    if shower_shape[0] != energy_shape[0]:
        raise ValueError("Number of showers and incident energies does not match.")
    if energy_shape not in [(shower_shape[0],), (shower_shape[0], 1)]:
        raise ValueError(f"Unexpected incident_energies shape: {energy_shape}")

    energies_raw_MeV = np.asarray(incident_ds[:], dtype=np.float64).reshape(-1)

energies_GeV = energies_raw_MeV / 1000.0
n_reference = int(len(energies_GeV))

if not np.isfinite(energies_GeV).all() or np.any(energies_GeV <= 0):
    raise ValueError("Incident energies contain non-finite or non-positive values.")

print("HDF5 keys:", keys)
print("Showers shape:", shower_shape, "dtype:", shower_dtype)
print("Incident energies shape:", energy_shape, "dtype:", energy_dtype)
print("Total reference showers:", n_reference)
print("Incident energy range (GeV):", (float(energies_GeV.min()), float(energies_GeV.max())))
print("Incident energy median (GeV):", float(np.median(energies_GeV)))

HDF5 keys: ['incident_energies', 'showers']
Showers shape: (12000, 6480) dtype: float32
Incident energies shape: (12000, 1) dtype: float32
Total reference showers: 12000
Incident energy range (GeV): (1.0007108764648438, 999.56425)
Incident energy median (GeV): 30.130296875


In [ ]:
# Cell 6 — Count showers in the three planned energy windows
window_rows = []
window_indices = {}

for lo, hi in ENERGY_WINDOWS_GEV:
    label = f"{lo:g}-{hi:g} GeV"
    idx = np.flatnonzero((energies_GeV >= lo) & (energies_GeV < hi))
    window_indices[label] = idx
    vals = energies_GeV[idx]
    window_rows.append({
        "energy_window": label,
        "low_inclusive_GeV": lo,
        "high_exclusive_GeV": hi,
        "available_reference_showers": int(len(idx)),
        "min_observed_GeV": float(vals.min()) if len(vals) else np.nan,
        "median_observed_GeV": float(np.median(vals)) if len(vals) else np.nan,
        "max_observed_GeV": float(vals.max()) if len(vals) else np.nan,
    })

window_counts = pd.DataFrame(window_rows)
display(window_counts)

print("Total events in selected windows:", int(window_counts.available_reference_showers.sum()))
print("Events outside selected windows:", int(n_reference - window_counts.available_reference_showers.sum()))

,energy_window,low_inclusive_GeV,high_exclusive_GeV,available_reference_showers,min_observed_GeV,median_observed_GeV,max_observed_GeV
0,8-12 GeV,8.0,12.0,726,8.002677,9.673570,11.983182
1,40-60 GeV,40.0,60.0,681,40.021535,48.974625,59.799023
2,80-120 GeV,80.0,120.0,696,80.158789,98.016445,119.982195


Total events in selected windows: 2103
Events outside selected windows: 9897


In [ ]:
# Cell 7 — Does each energy window support the planned split?
plan_rows = []
plans = {
    "smoke_light": SMOKE_LIGHT,
    "original_smoke": ORIGINAL_SMOKE,
    "proposed_300_150_300": PROPOSED_STUDY,
    "current_research_profile": RESEARCH_PROFILE,
}

for row in window_rows:
    n_available = row["available_reference_showers"]
    for plan_name, (n_cal, n_pilot, n_test) in plans.items():
        needed = n_cal + n_pilot + n_test
        plan_rows.append({
            "energy_window": row["energy_window"],
            "plan": plan_name,
            "calibration": n_cal,
            "pilot": n_pilot,
            "test": n_test,
            "total_required": needed,
            "available": n_available,
            "sufficient": bool(n_available >= needed and n_cal >= n_test),
        })

plan_table = pd.DataFrame(plan_rows)
display(plan_table)

proposed_ok = all(
    window_indices[f"{lo:g}-{hi:g} GeV"].size >= sum(PROPOSED_STUDY)
    for lo, hi in ENERGY_WINDOWS_GEV
) and N_CALIBRATION >= N_TEST

print("\nProposed 300/150/300 split available in every window:", proposed_ok)
if not proposed_ok:
    print("Do not start the planned study with these sizes yet. Use a larger approved reference file or predeclare smaller sizes.")
else:
    print("The selected file has enough distinct rows for the proposed split in all three windows.")

,energy_window,plan,calibration,pilot,test,total_required,available,sufficient
0,8-12 GeV,smoke_light,32,16,32,80,726,True
1,8-12 GeV,original_smoke,64,32,64,160,726,True
2,8-12 GeV,proposed_300_150_300,300,150,300,750,726,False
3,8-12 GeV,current_research_profile,2000,500,2000,4500,726,False
4,40-60 GeV,smoke_light,32,16,32,80,681,True
5,40-60 GeV,original_smoke,64,32,64,160,681,True
6,40-60 GeV,proposed_300_150_300,300,150,300,750,681,False
7,40-60 GeV,current_research_profile,2000,500,2000,4500,681,False
8,80-120 GeV,smoke_light,32,16,32,80,696,True
9,80-120 GeV,original_smoke,64,32,64,160,696,True



Proposed 300/150/300 split available in every window: False
Do not start the planned study with these sizes yet. Use a larger approved reference file or predeclare smaller sizes.


## 3. Inspect HDF5 shower values (sample or full streaming scan)

The shape/count check above does not read all 6,480 deposits for every shower. The next cell checks a sample by default. Set `SCAN_ALL_SHOWER_VALUES=True` in Cell 1 to scan every shower in chunks (more I/O, but more complete). It reports non-finite values, negative deposits, all-zero showers, and the sum of voxel deposits for inspected rows.

In [ ]:
# Cell 8 — Shower-array integrity check
rows_to_read = n_reference if SCAN_ALL_SHOWER_VALUES else min(SAMPLE_ROWS_TO_CHECK, n_reference)
chunk_size = 256
scanned = 0
nonfinite_values = 0
negative_values = 0
all_zero_showers = 0
sample_event_sums = []

with h5py.File(REFERENCE_HDF5, "r") as h5f:
    ds = h5f["showers"]
    for start in range(0, rows_to_read, chunk_size):
        stop = min(start + chunk_size, rows_to_read)
        block = np.asarray(ds[start:stop], dtype=np.float64)
        nonfinite_values += int((~np.isfinite(block)).sum())
        negative_values += int((block < 0).sum())
        all_zero_showers += int(np.sum(np.all(block == 0, axis=1)))
        if start == 0:
            sample_event_sums = block.sum(axis=1).tolist()
        scanned += len(block)

integrity_summary = pd.DataFrame([{
    "rows_scanned": scanned,
    "scan_scope": "all events" if SCAN_ALL_SHOWER_VALUES else "first sample rows only",
    "nonfinite_voxel_values": nonfinite_values,
    "negative_voxel_values": negative_values,
    "all_zero_showers_in_scanned_rows": all_zero_showers,
    "first_row_deposit_sum_MeV": sample_event_sums[0] if sample_event_sums else np.nan,
}])
display(integrity_summary)

if nonfinite_values or negative_values:
    raise ValueError("HDF5 shower-value integrity check found non-finite or negative deposits.")
print("Sample/full streaming shower-value check passed for the rows scanned.")

,rows_scanned,scan_scope,nonfinite_voxel_values,negative_voxel_values,all_zero_showers_in_scanned_rows,first_row_deposit_sum_MeV
0,256,first sample rows only,0,0,0,94183.497077


Sample/full streaming shower-value check passed for the rows scanned.


## 4. Verify native cell geometry and ordering

For Dataset 2, the intended regular native shape is `(45 layers, 16 angular bins, 9 radial bins)`, i.e. 6,480 cells. The expected flat-index rule is

`native_index = layer_index * 144 + angle_index * 9 + radial_index`.

Thus radial index changes fastest, then angle, then layer. This agrees with the standard CaloChallenge flattened-vector convention and the Dataset 2 config shape. The test below checks the original electron XML, the repository's `XMLHandler`, and the actual `GeomConverter` reshape/unreshape methods. It also constructs a cell-index lookup table.

Note: for Dataset 2 electrons, the model uses the regular `(45,16,9)` shape directly; it does not need the irregular-geometry GLaM remapping used for Dataset 1 photons/pions. We still test `GeomConverter` as an independent source-level ordering check.

In [ ]:
# Cell 9 — Parse the original XML geometry independently
xml_root = ET.parse(XML_PATH).getroot()
electron_node = next(
    (node for node in xml_root if node.attrib.get("name") == "electron"),
    None
)
if electron_node is None:
    raise ValueError("Could not find the electron entry in binning_dataset_2.xml")

xml_layers = []
for xml_position, layer_node in enumerate(list(electron_node)):
    edges = np.asarray(
        [float(value) for value in layer_node.attrib["r_edges"].split(",")],
        dtype=np.float64
    )
    n_angles = int(layer_node.attrib["n_bin_alpha"])
    layer_id = int(layer_node.attrib["id"])
    n_radial = len(edges) - 1
    if n_angles > 0 and n_radial > 0:
        xml_layers.append({
            "xml_position": xml_position,
            "layer_id": layer_id,
            "n_angles": n_angles,
            "n_radial": n_radial,
            "radial_edges": edges,
        })

n_layers = len(xml_layers)
voxel_count_from_xml = sum(x["n_angles"] * x["n_radial"] for x in xml_layers)

assert n_layers == 45, f"Expected 45 active electron layers, found {n_layers}"
assert all(x["n_angles"] == 16 for x in xml_layers), "Not every layer has 16 angular bins."
assert all(x["n_radial"] == 9 for x in xml_layers), "Not every layer has 9 radial bins."
assert voxel_count_from_xml == 6480, f"XML implies {voxel_count_from_xml} native cells, not 6480."

xml_layer_table = pd.DataFrame([
    {k: v for k, v in layer.items() if k != "radial_edges"}
    for layer in xml_layers
])
display(xml_layer_table.head())
display(xml_layer_table.tail())
print("PASS: XML defines", n_layers, "layers and", voxel_count_from_xml, "native cells.")

,xml_position,layer_id,n_angles,n_radial
0,0,0,16,9
1,1,1,16,9
2,2,2,16,9
3,3,3,16,9
4,4,4,16,9


,xml_position,layer_id,n_angles,n_radial
40,40,40,16,9
41,41,41,16,9
42,42,42,16,9
43,43,43,16,9
44,44,44,16,9


PASS: XML defines 45 layers and 6480 native cells.


In [ ]:
# Cell 10 — Compare XML with the repository's actual XMLHandler and GeomConverter
# Import the wrapper at the pinned checkout; it exposes the same utils used by the model.
os.environ["CALODIFF_DEVICE"] = "cpu"
sys.path.insert(0, str(CALO_ROOT))
import calodiff_probe as cp

cfg = cp._cfg("electron")
print("Electron config SHAPE_ORIG:", cfg.get("SHAPE_ORIG"))
print("Electron config SHAPE:", cfg.get("SHAPE"))
print("Electron config SHAPE_PAD:", cfg.get("SHAPE_PAD"))
print("Electron config ECUT (GeV):", cfg.get("ECUT"))
print("Electron config FILES:", cfg.get("FILES"))
print("Electron config EVAL:", cfg.get("EVAL"))

assert int(cfg["SHAPE_ORIG"][-1]) == 6480
assert list(map(int, cfg["SHAPE"][1:4])) == [45, 16, 9]
assert list(map(int, cfg["SHAPE_PAD"][2:5])) == [45, 16, 9]

xml_handler = cp.utils.XMLHandler("electron", str(XML_PATH))
geom_converter = cp.utils.GeomConverter(xml_handler)

active_positions = list(map(int, xml_handler.GetRelevantLayers()))
expected_positions = [x["xml_position"] for x in xml_layers]
assert active_positions == expected_positions, (
    "XMLHandler active-layer positions do not match XML active-layer positions."
)

for layer, pos in zip(xml_layers, active_positions):
    np.testing.assert_allclose(
        np.asarray(xml_handler.r_edges[pos], dtype=float),
        layer["radial_edges"], rtol=0, atol=1e-12
    )
    assert int(xml_handler.a_bins[pos]) == layer["n_angles"]

expected_boundaries = np.arange(0, 6481, 144, dtype=int)
np.testing.assert_array_equal(
    np.asarray(geom_converter.layer_boundaries, dtype=int),
    expected_boundaries
)
np.testing.assert_array_equal(
    np.asarray(geom_converter.lay_alphas, dtype=int),
    np.full(45, 16, dtype=int)
)
assert geom_converter.num_layers == 45

print("PASS: XMLHandler active layers match XML.")
print("PASS: XMLHandler radial edges and angular counts match XML.")
print("PASS: GeomConverter boundaries are 0, 144, ..., 6480.")
print("PASS: GeomConverter has 45 layers, each with 16 angular bins.")

Electron config SHAPE_ORIG: [-1, 6480]
Electron config SHAPE: [-1, 45, 16, 9, 1]
Electron config SHAPE_PAD: [-1, 1, 45, 16, 9]
Electron config ECUT (GeV): 1.51e-05
Electron config FILES: ['dataset_2_1.hdf5']
Electron config EVAL: ['dataset_2_2.hdf5']
PASS: XMLHandler active layers match XML.
PASS: XMLHandler radial edges and angular counts match XML.
PASS: GeomConverter boundaries are 0, 144, ..., 6480.
PASS: GeomConverter has 45 layers, each with 16 angular bins.


In [ ]:
# ============================================================
# FULL ELECTRON REFERENCE DATA INTEGRITY SCAN
# Scans ALL events and ALL 6480 voxels per event.
# ============================================================

import numpy as np
import pandas as pd
import h5py
import json
from pathlib import Path

CHUNK_ROWS = 256

# Electron threshold from the actual model configuration.
# Model config is in GeV; HDF5 energy deposits are in MeV.
ECUT_GEV = float(cfg["ECUT"])
ECUT_MEV = ECUT_GEV * 1000.0

print("Reference file:", REFERENCE_HDF5)
print("Analysis cutoff:", ECUT_MEV, "MeV")
print("Chunk size:", CHUNK_ROWS)


# ------------------------------------------------------------
# A. Initialize counters
# ------------------------------------------------------------

rows_scanned = 0

nonfinite_voxels = 0
negative_voxels = 0

nonfinite_energy_rows = 0
nonpositive_energy_rows = 0

all_zero_showers = 0
no_active_cells_after_cut = 0

rows_with_nonfinite = 0
rows_with_negative = 0

first_bad_nonfinite_rows = []
first_bad_negative_rows = []
first_zero_rows = []
first_no_active_rows = []

deposit_sums = []
responses = []
active_cell_counts = []

window_counts = {
    f"{lo:g}-{hi:g} GeV": 0
    for lo, hi in ENERGY_WINDOWS_GEV
}

max_voxel_energy = -np.inf
min_voxel_energy = np.inf

# ------------------------------------------------------------
# B. Verify and scan the complete HDF5 file
# ------------------------------------------------------------

with h5py.File(REFERENCE_HDF5, "r") as h5f:

    assert "showers" in h5f
    assert "incident_energies" in h5f

    showers_ds = h5f["showers"]
    incident_ds = h5f["incident_energies"]

    n_events, n_cells = showers_ds.shape

    assert n_cells == 6480, (
        f"Expected 6480 cells, found {n_cells}"
    )

    assert incident_ds.shape[0] == n_events, (
        "Shower count and incident-energy count do not match."
    )

    if not np.issubdtype(showers_ds.dtype, np.number):
        raise TypeError("Shower dataset is not numeric.")

    if not np.issubdtype(incident_ds.dtype, np.number):
        raise TypeError("Incident-energy dataset is not numeric.")

    print("\nStarting full scan...")
    print("Total showers:", n_events)
    print("Total voxel values:", n_events * n_cells)

    for start in range(0, n_events, CHUNK_ROWS):

        stop = min(start + CHUNK_ROWS, n_events)

        # Read only one chunk into memory.
        block = np.asarray(
            showers_ds[start:stop],
            dtype=np.float64
        )

        energy_MeV = np.asarray(
            incident_ds[start:stop],
            dtype=np.float64
        ).reshape(-1)

        # Check event alignment.
        assert len(block) == len(energy_MeV)

        # Check incident energy values.
        bad_energy = (
            ~np.isfinite(energy_MeV)
        )

        nonfinite_energy_rows += int(bad_energy.sum())

        bad_nonpositive = (
            np.isfinite(energy_MeV) &
            (energy_MeV <= 0)
        )

        nonpositive_energy_rows += int(
            bad_nonpositive.sum()
        )

        # Check every voxel.
        finite_mask = np.isfinite(block)
        negative_mask = block < 0

        nonfinite_voxels += int(
            (~finite_mask).sum()
        )

        negative_voxels += int(
            negative_mask.sum()
        )

        bad_nonfinite_row = (~finite_mask).any(axis=1)
        bad_negative_row = negative_mask.any(axis=1)

        rows_with_nonfinite += int(
            bad_nonfinite_row.sum()
        )

        rows_with_negative += int(
            bad_negative_row.sum()
        )

        # Record a few problematic row indices if present.
        for local in np.flatnonzero(bad_nonfinite_row):
            if len(first_bad_nonfinite_rows) < 20:
                first_bad_nonfinite_rows.append(
                    int(start + local)
                )

        for local in np.flatnonzero(bad_negative_row):
            if len(first_bad_negative_rows) < 20:
                first_bad_negative_rows.append(
                    int(start + local)
                )

        # Check completely empty showers.
        zero_rows = np.all(block == 0, axis=1)

        all_zero_showers += int(zero_rows.sum())

        for local in np.flatnonzero(zero_rows):
            if len(first_zero_rows) < 20:
                first_zero_rows.append(int(start + local))

        # Apply the electron analysis threshold.
        # Only deposits >= ECUT_MEV count as active.
        active_counts = np.sum(
            finite_mask & (block >= ECUT_MEV),
            axis=1
        )

        no_active = active_counts == 0

        no_active_cells_after_cut += int(
            no_active.sum()
        )

        for local in np.flatnonzero(no_active):
            if len(first_no_active_rows) < 20:
                first_no_active_rows.append(
                    int(start + local)
                )

        active_cell_counts.extend(
            active_counts.tolist()
        )

        # Calculate event-level deposited energy only
        # for rows with valid, non-negative deposits
        # and valid positive incident energies.
        valid_rows = (
            finite_mask.all(axis=1) &
            (~bad_negative_row) &
            np.isfinite(energy_MeV) &
            (energy_MeV > 0)
        )

        row_sums = np.sum(
            block[valid_rows],
            axis=1,
            dtype=np.float64
        )

        valid_energies = energy_MeV[valid_rows]

        deposit_sums.extend(row_sums.tolist())

        responses.extend(
            (row_sums / valid_energies).tolist()
        )

        # Track global finite voxel minimum and maximum.
        finite_values = block[finite_mask]

        if finite_values.size:
            max_voxel_energy = max(
                max_voxel_energy,
                float(finite_values.max())
            )

            min_voxel_energy = min(
                min_voxel_energy,
                float(finite_values.min())
            )

        # Count incident energies in each window.
        energy_GeV = energy_MeV / 1000.0

        for lo, hi in ENERGY_WINDOWS_GEV:

            label = f"{lo:g}-{hi:g} GeV"

            window_counts[label] += int(
                np.sum(
                    (energy_GeV >= lo) &
                    (energy_GeV < hi)
                )
            )

        rows_scanned += len(block)

        if rows_scanned % 2048 == 0 or rows_scanned == n_events:
            print(
                f"Scanned {rows_scanned:,} / {n_events:,} showers"
            )


# ------------------------------------------------------------
# C. Summarize full scan
# ------------------------------------------------------------

assert rows_scanned == n_events

response_array = np.asarray(responses)
deposit_array = np.asarray(deposit_sums)
active_array = np.asarray(active_cell_counts)

integrity_summary = pd.DataFrame([{
    "total_reference_showers": n_events,
    "rows_scanned": rows_scanned,
    "cells_per_shower": n_cells,
    "total_voxel_values_scanned": n_events * n_cells,
    "nonfinite_voxel_values": nonfinite_voxels,
    "negative_voxel_values": negative_voxels,
    "rows_with_nonfinite_values": rows_with_nonfinite,
    "rows_with_negative_values": rows_with_negative,
    "nonfinite_incident_energy_rows": nonfinite_energy_rows,
    "nonpositive_incident_energy_rows": nonpositive_energy_rows,
    "completely_zero_showers": all_zero_showers,
    "showers_with_no_cell_above_cutoff": no_active_cells_after_cut,
    "cutoff_MeV": ECUT_MEV,
    "minimum_finite_voxel_MeV": min_voxel_energy,
    "maximum_finite_voxel_MeV": max_voxel_energy,
    "valid_response_rows": len(response_array),
}])

print("\n========== FULL INTEGRITY SUMMARY ==========")
display(integrity_summary)


# ------------------------------------------------------------
# D. Response and deposited-energy descriptive statistics
# ------------------------------------------------------------

if len(response_array) > 0:

    response_table = pd.DataFrame([{
        "mean_response": np.mean(response_array),
        "std_response": np.std(response_array, ddof=1),
        "response_q01": np.quantile(response_array, 0.01),
        "response_q05": np.quantile(response_array, 0.05),
        "response_median": np.quantile(response_array, 0.50),
        "response_q95": np.quantile(response_array, 0.95),
        "response_q99": np.quantile(response_array, 0.99),
        "maximum_response": np.max(response_array),
        "mean_deposit_MeV": np.mean(deposit_array),
        "median_deposit_MeV": np.median(deposit_array),
        "mean_active_cells": np.mean(active_array),
    }])

    print("\nDescriptive statistics:")
    display(response_table)

else:
    response_table = pd.DataFrame()
    print("WARNING: No valid showers available for response statistics.")


# ------------------------------------------------------------
# E. Energy-window counts
# ------------------------------------------------------------

window_integrity = pd.DataFrame([
    {
        "energy_window": window,
        "reference_showers": count
    }
    for window, count in window_counts.items()
])

print("\nEnergy-window counts after full scan:")
display(window_integrity)


# ------------------------------------------------------------
# F. Print suspicious row indices
# ------------------------------------------------------------

print("\nFirst rows with non-finite deposits:",
      first_bad_nonfinite_rows)

print("First rows with negative deposits:",
      first_bad_negative_rows)

print("First completely empty showers:",
      first_zero_rows)

print("First showers with no cells above cutoff:",
      first_no_active_rows)


# ------------------------------------------------------------
# G. Decide whether the file passes basic integrity
# ------------------------------------------------------------

hard_failures = []

if rows_scanned != n_events:
    hard_failures.append("Not all shower rows were scanned.")

if nonfinite_voxels > 0:
    hard_failures.append("Non-finite voxel values detected.")

if negative_voxels > 0:
    hard_failures.append("Negative voxel deposits detected.")

if nonfinite_energy_rows > 0:
    hard_failures.append("Non-finite incident energies detected.")

if nonpositive_energy_rows > 0:
    hard_failures.append("Non-positive incident energies detected.")

FULL_INTEGRITY_PASSED = len(hard_failures) == 0

print("\n========== FINAL RESULT ==========")

if FULL_INTEGRITY_PASSED:
    print("PASS: No invalid numerical values found in the full dataset.")
else:
    print("FAIL: Integrity problems detected.")
    for item in hard_failures:
        print("-", item)

# Empty showers and no-active-cell showers are warnings,
# not automatically hard failures.
if all_zero_showers > 0:
    print("WARNING: Completely empty showers require inspection.")

if no_active_cells_after_cut > 0:
    print("WARNING: Some showers have no cells above the analysis cutoff.")

print("\nReference provenance is NOT established by this scan.")


# ------------------------------------------------------------
# H. Export full integrity results
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

integrity_summary.to_csv(
    OUTPUT_DIR / "electron_full_integrity_summary.csv",
    index=False
)

response_table.to_csv(
    OUTPUT_DIR / "electron_full_integrity_response_statistics.csv",
    index=False
)

window_integrity.to_csv(
    OUTPUT_DIR / "electron_full_integrity_window_counts.csv",
    index=False
)

full_integrity_report = {
    "reference_file": str(REFERENCE_HDF5),
    "reference_sha256": REFERENCE_SHA256,
    "rows_scanned": int(rows_scanned),
    "cells_per_shower": int(n_cells),
    "total_voxel_values_scanned": int(n_events * n_cells),
    "cutoff_MeV": float(ECUT_MEV),
    "nonfinite_voxels": int(nonfinite_voxels),
    "negative_voxels": int(negative_voxels),
    "nonfinite_energy_rows": int(nonfinite_energy_rows),
    "nonpositive_energy_rows": int(nonpositive_energy_rows),
    "all_zero_showers": int(all_zero_showers),
    "no_active_cells_after_cut": int(no_active_cells_after_cut),
    "first_bad_nonfinite_rows": first_bad_nonfinite_rows,
    "first_bad_negative_rows": first_bad_negative_rows,
    "first_zero_rows": first_zero_rows,
    "first_no_active_rows": first_no_active_rows,
    "energy_window_counts": window_counts,
    "full_integrity_passed": bool(FULL_INTEGRITY_PASSED),
    "reference_provenance_verified": False,
}

(OUTPUT_DIR / "electron_full_integrity_report.json").write_text(
    json.dumps(full_integrity_report, indent=2)
)

print("\nSaved full integrity outputs to:", OUTPUT_DIR)


Reference file: /content/electron-preflight-gen-hep-notebooks/capstone/calodiffusion/data/calochallenge_ds2.hdf5
Analysis cutoff: 0.015099999999999999 MeV
Chunk size: 256

Starting full scan...
Total showers: 12000
Total voxel values: 77760000
Scanned 2,048 / 12,000 showers
Scanned 4,096 / 12,000 showers
Scanned 6,144 / 12,000 showers
Scanned 8,192 / 12,000 showers
Scanned 10,240 / 12,000 showers
Scanned 12,000 / 12,000 showers

========== FULL INTEGRITY SUMMARY ==========


,total_reference_showers,rows_scanned,cells_per_shower,total_voxel_values_scanned,nonfinite_voxel_values,negative_voxel_values,rows_with_nonfinite_values,rows_with_negative_values,nonfinite_incident_energy_rows,nonpositive_incident_energy_rows,completely_zero_showers,showers_with_no_cell_above_cutoff,cutoff_MeV,minimum_finite_voxel_MeV,maximum_finite_voxel_MeV,valid_response_rows
0,12000,12000,6480,77760000,0,0,0,0,0,0,0,0,0.0151,0.0,4355.755371,12000



Descriptive statistics:


,mean_response,std_response,response_q01,response_q05,response_median,response_q95,response_q99,maximum_response,mean_deposit_MeV,median_deposit_MeV,mean_active_cells
0,0.782574,0.112717,0.401162,0.591313,0.782429,0.966857,1.07559,1.877494,110693.780488,23510.212745,1571.452917



Energy-window counts after full scan:


,energy_window,reference_showers
0,8-12 GeV,726
1,40-60 GeV,681
2,80-120 GeV,696



First rows with non-finite deposits: []
First rows with negative deposits: []
First completely empty showers: []
First showers with no cells above cutoff: []

========== FINAL RESULT ==========
PASS: No invalid numerical values found in the full dataset.

Reference provenance is NOT established by this scan.

Saved full integrity outputs to: /content/electron_preflight_outputs


In [ ]:
# ============================================================
# ELECTRON PREFLIGHT: EMPTY AND LOW-OCCUPANCY SHOWER AUDIT
# Scans all reference showers in chunks.
# ============================================================

import numpy as np
import pandas as pd
import h5py
from pathlib import Path

# Electron cutoff:
# 15.1 keV = 0.0151 MeV
ECUT_GEV = float(cfg["ECUT"])
ECUT_MEV = ECUT_GEV * 1000.0

ENERGY_WINDOWS_GEV = [
    (8, 12),
    (40, 60),
    (80, 120)
]

CHUNK_ROWS = 256

print("Electron analysis cutoff:", ECUT_MEV, "MeV")
print("Reference file:", REFERENCE_HDF5)


# ------------------------------------------------------------
# 1. Initialize counters
# ------------------------------------------------------------

total_events = 0
total_empty = 0
total_no_active = 0
total_nonempty_no_active = 0

window_results = []
flagged_rows = []


# ------------------------------------------------------------
# 2. Scan the complete reference file
# ------------------------------------------------------------

with h5py.File(REFERENCE_HDF5, "r") as h5f:

    showers_ds = h5f["showers"]
    energies_ds = h5f["incident_energies"]

    n_events, n_cells = showers_ds.shape

    assert n_cells == 6480, (
        f"Expected 6480 cells, found {n_cells}"
    )

    for start in range(0, n_events, CHUNK_ROWS):

        stop = min(start + CHUNK_ROWS, n_events)

        showers = np.asarray(
            showers_ds[start:stop],
            dtype=np.float64
        )

        incident_GeV = np.asarray(
            energies_ds[start:stop],
            dtype=np.float64
        ).reshape(-1) / 1000.0

        # A completely empty shower:
        # every cell is exactly zero.
        empty = np.all(showers == 0, axis=1)

        # Count cells with deposits at or above 15.1 keV.
        active_counts = np.sum(
            np.isfinite(showers) &
            (showers >= ECUT_MEV),
            axis=1
        )

        # No active cells after applying the cutoff.
        no_active = active_counts == 0

        # Non-empty but all deposits remain below the cutoff.
        nonempty_no_active = no_active & (~empty)

        total_events += len(showers)
        total_empty += int(empty.sum())
        total_no_active += int(no_active.sum())
        total_nonempty_no_active += int(
            nonempty_no_active.sum()
        )

        # Record individual flagged showers.
        for i in np.flatnonzero(no_active):

            flagged_rows.append({
                "reference_row": start + int(i),
                "incident_energy_GeV": incident_GeV[i],
                "completely_empty": bool(empty[i]),
                "active_cells_above_cutoff": int(
                    active_counts[i]
                ),
                "nonzero_cells_below_cutoff": int(
                    np.count_nonzero(showers[i])
                ),
                "total_deposit_MeV": float(
                    np.sum(showers[i])
                )
            })

        if stop % 2048 == 0 or stop == n_events:
            print(f"Scanned {stop:,}/{n_events:,} showers")


# ------------------------------------------------------------
# 3. Overall summary
# ------------------------------------------------------------

overall_summary = pd.DataFrame([{
    "total_showers": total_events,
    "completely_empty_showers": total_empty,
    "showers_with_no_active_cells": total_no_active,
    "nonempty_but_no_active_cells": total_nonempty_no_active,
    "empty_shower_percentage": (
        100 * total_empty / total_events
    ),
    "no_active_cell_percentage": (
        100 * total_no_active / total_events
    )
}])

print("\nOVERALL RESULTS")
display(overall_summary)


# ------------------------------------------------------------
# 4. Break down results by energy window
# ------------------------------------------------------------

# Explicitly define columns so that even if flagged_rows is empty,
# we don't encounter a KeyError when accessing the columns.
flagged_df = pd.DataFrame(
    flagged_rows,
    columns=[
        "reference_row",
        "incident_energy_GeV",
        "completely_empty",
        "active_cells_above_cutoff",
        "nonzero_cells_below_cutoff",
        "total_deposit_MeV"
    ]
)

with h5py.File(REFERENCE_HDF5, "r") as h5f:

    energies_GeV = np.asarray(
        h5f["incident_energies"][:],
        dtype=np.float64
    ).reshape(-1) / 1000.0

for lo, hi in ENERGY_WINDOWS_GEV:

    in_window = (
        (energies_GeV >= lo) &
        (energies_GeV < hi)
    )

    total_in_window = int(in_window.sum())

    if total_in_window == 0:
        window_results.append({
            "energy_window_GeV": f"{lo}-{hi}",
            "total_showers": 0,
            "empty_showers": 0,
            "no_active_showers": 0,
            "nonempty_no_active": 0,
            "no_active_percentage": np.nan
        })
        continue

    subset = flagged_df[
        (flagged_df["incident_energy_GeV"] >= lo) &
        (flagged_df["incident_energy_GeV"] < hi)
    ]

    empty_count = int(subset["completely_empty"].sum())
    no_active_count = len(subset)

    window_results.append({
        "energy_window_GeV": f"{lo}-{hi}",
        "total_showers": total_in_window,
        "empty_showers": empty_count,
        "no_active_showers": no_active_count,
        "nonempty_no_active": (
            no_active_count - empty_count
        ),
        "no_active_percentage": (
            100 * no_active_count / total_in_window
        )
    })

window_summary = pd.DataFrame(window_results)

print("\nENERGY-WINDOW RESULTS")
display(window_summary)


# ------------------------------------------------------------
# 5. Show individual flagged showers
# ------------------------------------------------------------

print("\nINDIVIDUAL FLAGGED SHOWERS")

if len(flagged_df) == 0:
    print("No completely empty or zero-active showers found.")
else:
    display(flagged_df.sort_values(
        "incident_energy_GeV"
    ))


# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

output_dir = Path(
    globals().get(
        "OUTPUT_DIR",
        "/content/electron_preflight_outputs"
    )
)

output_dir.mkdir(parents=True, exist_ok=True)

overall_summary.to_csv(
    output_dir / "electron_empty_shower_summary.csv",
    index=False
)

window_summary.to_csv(
    output_dir / "electron_empty_shower_by_energy.csv",
    index=False
)

flagged_df.to_csv(
    output_dir / "electron_flagged_empty_or_low_activity.csv",
    index=False
)

print("\nSaved all three reports to:")
print(output_dir)

Electron analysis cutoff: 0.015099999999999999 MeV
Reference file: /content/electron-preflight-gen-hep-notebooks/capstone/calodiffusion/data/calochallenge_ds2.hdf5
Scanned 2,048/12,000 showers
Scanned 4,096/12,000 showers
Scanned 6,144/12,000 showers
Scanned 8,192/12,000 showers
Scanned 10,240/12,000 showers
Scanned 12,000/12,000 showers

OVERALL RESULTS


,total_showers,completely_empty_showers,showers_with_no_active_cells,nonempty_but_no_active_cells,empty_shower_percentage,no_active_cell_percentage
0,12000,0,0,0,0.0,0.0



ENERGY-WINDOW RESULTS


,energy_window_GeV,total_showers,empty_showers,no_active_showers,nonempty_no_active,no_active_percentage
0,8-12,726,0,0,0,0.0
1,40-60,681,0,0,0,0.0
2,80-120,696,0,0,0,0.0



INDIVIDUAL FLAGGED SHOWERS
No completely empty or zero-active showers found.

Saved all three reports to:
/content/electron_preflight_outputs


In [ ]:
# Cell 11 — Artificial-number (sentinel) reshape and reverse-reshape test
# Every native position contains its own index: [0, 1, 2, ..., 6479].
sentinel = torch.arange(6480, dtype=torch.float32).reshape(1, 6480)
layer_tensors = geom_converter.reshape(sentinel)

assert len(layer_tensors) == 45
sentinel_rows = []

for layer_index, layer_tensor in enumerate(layer_tensors):
    start = layer_index * 144
    stop = start + 144
    expected = np.arange(start, stop, dtype=np.float32).reshape(1, 16, 9)
    actual = layer_tensor.detach().cpu().numpy()
    assert actual.shape == (1, 16, 9), (
        f"Layer {layer_index}: unexpected shape {actual.shape}"
    )
    np.testing.assert_array_equal(actual, expected)
    sentinel_rows.append({
        "layer_index": layer_index,
        "expected_native_start": start,
        "expected_native_end_inclusive": stop - 1,
        "actual_shape": str(actual.shape),
        "result": "PASS"
    })

sentinel_table = pd.DataFrame(sentinel_rows)
display(sentinel_table)

restored = geom_converter.unreshape(layer_tensors)
assert tuple(restored.shape) == (1, 6480)
assert torch.equal(restored.cpu(), sentinel.cpu())

# Also test the model's direct Dataset 2 reshape convention.
direct_grid = sentinel.numpy().reshape(1, 45, 16, 9)
direct_roundtrip = direct_grid.reshape(1, 6480)
np.testing.assert_array_equal(direct_roundtrip, sentinel.numpy())

print("PASS: all 45 layer slices match the expected contiguous native indices.")
print("PASS: GeomConverter reshape -> unreshape preserves all 6,480 indices.")
print("PASS: direct Dataset 2 (45,16,9) reshape round-trip preserves ordering.")

,layer_index,expected_native_start,expected_native_end_inclusive,actual_shape,result
0,0,0,143,"(1, 16, 9)",PASS
1,1,144,287,"(1, 16, 9)",PASS
2,2,288,431,"(1, 16, 9)",PASS
3,3,432,575,"(1, 16, 9)",PASS
4,4,576,719,"(1, 16, 9)",PASS
5,5,720,863,"(1, 16, 9)",PASS
6,6,864,1007,"(1, 16, 9)",PASS
7,7,1008,1151,"(1, 16, 9)",PASS
8,8,1152,1295,"(1, 16, 9)",PASS
9,9,1296,1439,"(1, 16, 9)",PASS


PASS: all 45 layer slices match the expected contiguous native indices.
PASS: GeomConverter reshape -> unreshape preserves all 6,480 indices.
PASS: direct Dataset 2 (45,16,9) reshape round-trip preserves ordering.


In [ ]:
# Cell 12 — Build a complete native index -> (layer, angle, radial) map
native_rows = []
for layer_index, layer in enumerate(xml_layers):
    edges = layer["radial_edges"]
    for angle_index in range(layer["n_angles"]):
        for radial_index in range(layer["n_radial"]):
            native_index = layer_index * 144 + angle_index * 9 + radial_index
            native_rows.append({
                "native_index": native_index,
                "layer_index": layer_index,
                "xml_layer_id": layer["layer_id"],
                "angle_index": angle_index,
                "radial_index": radial_index,
                "r_low_XML": edges[radial_index],
                "r_high_XML": edges[radial_index + 1],
                "r_center_XML": 0.5 * (edges[radial_index] + edges[radial_index + 1]),
            })

native_map = pd.DataFrame(native_rows)
assert len(native_map) == 6480
assert native_map.native_index.min() == 0
assert native_map.native_index.max() == 6479
assert native_map.native_index.is_unique

display(native_map.head(12))
display(native_map.tail(12))
print("PASS: complete 6,480-row native-cell map constructed.")

,native_index,layer_index,xml_layer_id,angle_index,radial_index,r_low_XML,r_high_XML,r_center_XML
0,0,0,0,0,0,0.00,4.65,2.325
1,1,0,0,0,1,4.65,9.30,6.975
2,2,0,0,0,2,9.30,13.95,11.625
3,3,0,0,0,3,13.95,18.60,16.275
4,4,0,0,0,4,18.60,23.25,20.925
5,5,0,0,0,5,23.25,27.90,25.575
6,6,0,0,0,6,27.90,32.55,30.225
7,7,0,0,0,7,32.55,37.20,34.875
8,8,0,0,0,8,37.20,41.85,39.525
9,9,0,0,1,0,0.00,4.65,2.325


,native_index,layer_index,xml_layer_id,angle_index,radial_index,r_low_XML,r_high_XML,r_center_XML
6468,6468,44,44,14,6,27.90,32.55,30.225
6469,6469,44,44,14,7,32.55,37.20,34.875
6470,6470,44,44,14,8,37.20,41.85,39.525
6471,6471,44,44,15,0,0.00,4.65,2.325
6472,6472,44,44,15,1,4.65,9.30,6.975
6473,6473,44,44,15,2,9.30,13.95,11.625
6474,6474,44,44,15,3,13.95,18.60,16.275
6475,6475,44,44,15,4,18.60,23.25,20.925
6476,6476,44,44,15,5,23.25,27.90,25.575
6477,6477,44,44,15,6,27.90,32.55,30.225


PASS: complete 6,480-row native-cell map constructed.


## 5. Create a fixed candidate partition (if sample counts permit)

This creates one deterministic calibration/pilot/test partition per energy window. It is **not yet wired automatically into the main validation notebook**; that notebook must be edited to load this CSV and keep these same rows across all three generation seeds. The point of this file is to establish the fixed row indices now, before looking at fidelity results.

In [ ]:
# Cell 14 — Export fixed candidate event partitions
partition_rows = []
partition_status = []
required = N_CALIBRATION + N_PILOT + N_TEST

if N_CALIBRATION < N_TEST:
    raise ValueError("Calibration donors must be at least as numerous as test events for the current scale-only baseline.")

for lo, hi in ENERGY_WINDOWS_GEV:
    label = f"{lo:g}-{hi:g}GeV"
    candidates = np.asarray(window_indices[f"{lo:g}-{hi:g} GeV"], dtype=int)
    ok = len(candidates) >= required
    partition_status.append({
        "energy_window": label,
        "available": int(len(candidates)),
        "required": int(required),
        "can_create_partition": bool(ok),
    })
    if not ok:
        continue

    # Stable per-window RNG, independent of future generation seeds.
    seed_bytes = hashlib.sha256(f"{PARTITION_SEED}|electron|{label}".encode()).digest()[:4]
    local_seed = int.from_bytes(seed_bytes, "big")
    rng = np.random.default_rng(local_seed)
    chosen = rng.permutation(candidates)[:required]

    a = N_CALIBRATION
    b = N_CALIBRATION + N_PILOT
    split_indices = {
        "calibration": chosen[:a],
        "pilot": chosen[a:b],
        "test": chosen[b:],
    }
    for split_name, row_indices in split_indices.items():
        for row_idx in row_indices:
            partition_rows.append({
                "particle": "electron",
                "energy_window": label,
                "partition": split_name,
                "hdf5_row": int(row_idx),
                "event_energy_GeV": float(energies_GeV[row_idx]),
                "event_id": f"{REFERENCE_FILE_ID}:{int(row_idx)}",
                "partition_seed": PARTITION_SEED,
            })

partition_status_df = pd.DataFrame(partition_status)
display(partition_status_df)

if all(x["can_create_partition"] for x in partition_status):
    fixed_partitions = pd.DataFrame(partition_rows)
    partition_csv = OUTPUT_DIR / "electron_fixed_candidate_partitions.csv"
    fixed_partitions.to_csv(partition_csv, index=False)
    print("Saved fixed candidate partitions:", partition_csv)
    print(f"Rows saved: {len(fixed_partitions)}")
else:
    fixed_partitions = pd.DataFrame()
    print("No fixed partition CSV created: one or more energy windows lack the requested event count.")

,energy_window,available,required,can_create_partition
0,8-12GeV,726,750,False
1,40-60GeV,681,750,False
2,80-120GeV,696,750,False


No fixed partition CSV created: one or more energy windows lack the requested event count.


In [ ]:
# Cell 15 — Compact readiness summary
readiness_rows = []
for r in window_rows:
    n = int(r["available_reference_showers"])
    readiness_rows.append({
        "window": r["energy_window"],
        "available": n,
        "80_event_light_smoke": n >= sum(SMOKE_LIGHT),
        "160_event_original_smoke": n >= sum(ORIGINAL_SMOKE),
        "750_event_proposed_study": n >= sum(PROPOSED_STUDY),
        "4500_event_research_profile": n >= sum(RESEARCH_PROFILE),
    })
readiness_df = pd.DataFrame(readiness_rows)
display(readiness_df)

print("\nReference file:", REFERENCE_HDF5)
print("Total events:", n_reference)
print("Voxel count per shower:", shower_shape[1])
print("Geometry:", f"{n_layers} x 16 x 9 = {voxel_count_from_xml}")
print("Electron ECUT from model config (GeV):", cfg.get("ECUT"))
print("Electron ECUT from model config (keV):", float(cfg.get("ECUT")) * 1e6)
print("Reference provenance confirmed by guide? No automatic determination is possible here.")

if proposed_ok:
    print("\nPRE-FLIGHT: EVENT COUNTS SUPPORT THE PROPOSED 300/150/300 SPLIT.")
else:
    print("\nPRE-FLIGHT: EVENT COUNTS DO NOT SUPPORT 300/150/300 IN EVERY WINDOW.")
print("Native geometry and ordering tests should be reviewed independently from event-count readiness.")

,window,available,80_event_light_smoke,160_event_original_smoke,750_event_proposed_study,4500_event_research_profile
0,8-12 GeV,726,True,True,False,False
1,40-60 GeV,681,True,True,False,False
2,80-120 GeV,696,True,True,False,False



Reference file: /content/electron-preflight-gen-hep-notebooks/capstone/calodiffusion/data/calochallenge_ds2.hdf5
Total events: 12000
Voxel count per shower: 6480
Geometry: 45 x 16 x 9 = 6480
Electron ECUT from model config (GeV): 1.51e-05
Electron ECUT from model config (keV): 15.1
Reference provenance confirmed by guide? No automatic determination is possible here.

PRE-FLIGHT: EVENT COUNTS DO NOT SUPPORT 300/150/300 IN EVERY WINDOW.
Native geometry and ordering tests should be reviewed independently from event-count readiness.


In [ ]:
# Cell 16 — Save tables, geometry map, and a machine-readable report
import csv
import pandas as pd
import json

# Convert window_counts dictionary to a DataFrame so it has the to_csv method
window_counts_df = pd.DataFrame(list(window_counts.items()), columns=["energy_window", "available_reference_showers"])
window_counts_df.to_csv(OUTPUT_DIR / "electron_reference_window_counts.csv", index=False)

plan_table.to_csv(OUTPUT_DIR / "electron_split_capacity.csv", index=False)
integrity_summary.to_csv(OUTPUT_DIR / "electron_hdf5_integrity_sample.csv", index=False)
xml_layer_table.to_csv(OUTPUT_DIR / "electron_xml_layer_geometry.csv", index=False)
native_map.to_csv(OUTPUT_DIR / "electron_native_cell_map.csv", index=False)
sentinel_table.to_csv(OUTPUT_DIR / "electron_native_order_sentinel_test.csv", index=False)
readiness_df.to_csv(OUTPUT_DIR / "electron_preflight_readiness.csv", index=False)

report = {
    "repository_revision": actual_sha,
    "reference_file": str(REFERENCE_HDF5),
    "reference_file_size_bytes": REFERENCE_HDF5.stat().st_size,
    "reference_file_sha256": REFERENCE_SHA256,
    "reference_event_count": n_reference,
    "showers_shape": list(shower_shape),
    "incident_energies_shape": list(energy_shape),
    "energy_unit_in_file": INPUT_UNIT,
    "energy_windows_GeV": [list(x) for x in ENERGY_WINDOWS_GEV],
    "window_counts": window_rows,
    "proposed_split": {"calibration": N_CALIBRATION, "pilot": N_PILOT, "test": N_TEST},
    "proposed_split_sufficient_all_windows": bool(proposed_ok),
    "geometry": {
        "active_layers": n_layers,
        "angular_bins_per_layer": sorted(set(x["n_angles"] for x in xml_layers)),
        "radial_bins_per_layer": sorted(set(x["n_radial" ] for x in xml_layers)),
        "native_cell_count": voxel_count_from_xml,
        "geom_converter_boundaries": np.asarray(geom_converter.layer_boundaries).tolist(),
        "sentinel_reshape_passed": True,
        "sentinel_unreshape_roundtrip_passed": True,
        "native_index_formula": "index = layer_index*144 + angle_index*9 + radial_index",
    },
    "electron_model_config": {
        "SHAPE_ORIG": cfg.get("SHAPE_ORIG"),
        "SHAPE": cfg.get("SHAPE"),
        "SHAPE_PAD": cfg.get("SHAPE_PAD"),
        "ECUT_GeV": float(cfg.get("ECUT")),
        "FILES": cfg.get("FILES"),
        "EVAL": cfg.get("EVAL"),
    },
    "provenance_note": "This notebook cannot prove whether the selected reference rows were excluded from checkpoint training or selection. Confirm the exact HDF5 identity and provenance with the guide.",
}
(OUTPUT_DIR / "electron_preflight_report.json").write_text(json.dumps(report, indent=2))

print("Saved preflight outputs to:", OUTPUT_DIR)
print("Key files: electron_preflight_report.json, electron_reference_window_counts.csv, electron_native_cell_map.csv")

Saved preflight outputs to: /content/electron_preflight_outputs
Key files: electron_preflight_report.json, electron_reference_window_counts.csv, electron_native_cell_map.csv


In [ ]:
# Cell 17 — Download the preflight results bundle
archive_path = Path("/content/electron_preflight_results.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.glob("*")):
        if path.is_file():
            zf.write(path, arcname=path.name)

print("Created:", archive_path, "| MiB:", round(archive_path.stat().st_size / 2**20, 2))
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    print("Not running in Colab. Download the ZIP from the file browser:", archive_path)

Created: /content/electron_preflight_results.zip | MiB: 0.03


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# How to interpret the preflight

## Event-count decision

- **750 or more in every window:** the proposed `300 calibration + 150 pilot + 300 test` design fits the inspected file.
- **80–749 in a window:** the file may support a smaller smoke or custom study, but not the proposed sample size. Decide and predeclare reduced counts before looking at fidelity results.
- **Fewer than 80 in a window:** the current file does not support even the light smoke split for that energy window; obtain the approved evaluation file or revise the windows before proceeding.
- **4,500 or more in every window:** the current validation notebook's research profile sample counts fit, although other compute/runtime limits still apply.

Do not combine calibration, pilot and test events or reuse test events to fill calibration. Calibration donors must be at least as numerous as test events for the current scale-only comparison.

## Native-ordering decision

A passing result establishes that:
1. XML declares 45 active layers, each with 16 angular and 9 radial bins.
2. Total native cells = 6,480.
3. The pinned repository's `XMLHandler` agrees with the XML.
4. The pinned repository's `GeomConverter` reshapes contiguous native segments into `(16,9)` per layer and unreshapes them back exactly.
5. The validation config expects the regular Dataset 2 shape `(45,16,9)`.

The flat index is therefore interpreted as:

`index = layer_index * 144 + angle_index * 9 + radial_index`.

The HDF5 file itself has no separate coordinate label for every flattened element; the semantic mapping comes from the Dataset 2 geometry definition and the repository's documented loader/converter convention. The sentinel test proves software consistency, not detector physics.

## Still required before the three-seed fidelity study

1. Confirm with your guide that the chosen reference file is the intended evaluation split and was not used for checkpoint training or selection.
2. Confirm the event counts support the chosen split; use the exported fixed partition CSV in the main validation notebook.
3. Modify the main validation notebook so the partition seed is fixed independently from the three generation seeds. The current implementation derives partition selection from `CFG.seed`; simply changing the seed would change the reference events too.
4. Use the same source revision, checkpoint, reference HDF5 hash, analysis cutoff, and fixed calibration/pilot/test event IDs across the three runs. Change only the generation seed.

This preflight does not generate showers and does not declare the reference file independent.